# Python Implementation Here

In [1]:
# libraries
import pandas as pd
import warnings
import numpy as np
import time
# disable warnings
warnings.filterwarnings("ignore")

In [4]:
data = pd.read_csv('data/taxi_tripdata.csv')
data.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,1.0,2021-07-01 00:30:52,2021-07-01 00:35:36,N,1.0,74,168,1.0,1.20,6.0,0.5,0.5,0.00,0.0,NaN,0.3,7.30,2.0,1.0,0.0
1,2.0,2021-07-01 00:25:36,2021-07-01 01:01:31,N,1.0,116,265,2.0,13.69,42.0,0.5,0.5,0.00,0.0,NaN,0.3,43.30,2.0,1.0,0.0
2,2.0,2021-07-01 00:05:58,2021-07-01 00:12:00,N,1.0,97,33,1.0,0.95,6.5,0.5,0.5,2.34,0.0,NaN,0.3,10.14,1.0,1.0,0.0
3,2.0,2021-07-01 00:41:40,2021-07-01 00:47:23,N,1.0,74,42,1.0,1.24,6.5,0.5,0.5,0.00,0.0,NaN,0.3,7.80,2.0,1.0,0.0
4,2.0,2021-07-01 00:51:32,2021-07-01 00:58:46,N,1.0,42,244,1.0,1.10,7.0,0.5,0.5,0.00,0.0,NaN,0.3,8.30,2.0,1.0,0.0


# Data Processing

In [48]:
import pandas as pd
import numpy as np
import time
import psutil
import os

# Track time + resource usage
def log_time(name, start_time):
    elapsed = time.time() - start_time
    process = psutil.Process(os.getpid())
    memory = process.memory_info().rss / (1024 * 1024)  # in MB
    cpu = psutil.cpu_percent(interval=0.1)
    print(f"{name} took {elapsed:.2f} seconds | Memory: {memory:.2f} MB | CPU: {cpu:.2f}%\n")

# --- 1. Load Data ---
start = time.time()
df = pd.read_csv("data/taxi_data.csv")
log_time("Load CSV", start)

# --- 2. Data Cleaning ---
start = time.time()

# Clean store_and_fwd_flag
df['store_and_fwd_flag'] = df['store_and_fwd_flag'].replace('', np.nan).fillna('N')

# Clean passenger_count
df['passenger_count'] = df['passenger_count'].replace(0, np.nan).fillna(1)

# Clamp trip_distance to [0, 100] or set to NaN
df['trip_distance'] = df['trip_distance'].apply(lambda x: x if 0 <= x <= 100 else np.nan)

# Replace NULLs or negative values in monetary columns
monetary_cols = [
    'fare_amount', 'extra', 'mta_tax', 'tip_amount',
    'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge'
]
df[monetary_cols] = df[monetary_cols].applymap(lambda x: max(x, 0) if pd.notna(x) else 0)

# Convert datetimes
df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'], errors='coerce')
df['lpep_dropoff_datetime'] = pd.to_datetime(df['lpep_dropoff_datetime'], errors='coerce')

# Drop invalid timestamp rows
df = df.dropna(subset=['lpep_pickup_datetime', 'lpep_dropoff_datetime'])
df = df[df['lpep_dropoff_datetime'] > df['lpep_pickup_datetime']]

# Compute trip duration
df['trip_duration_minutes'] = (df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']).dt.total_seconds() / 60

log_time("Data Cleaning and Transformation", start)

# --- 3. Save Cleaned Data ---
start = time.time()
df = df.drop(columns=["ehail_fee"])
df = df.dropna()
df.to_csv("python_output/taxi_trips_cleaned.csv", index=False)
log_time("Save Cleaned CSV", start)

# --- 4. Data Quality Checks ---
start = time.time()
issue_counts = {
    "Invalid pickup/dropoff time": (df['lpep_dropoff_datetime'] <= df['lpep_pickup_datetime']).sum(),
    "Null pickup or dropoff datetime": df[['lpep_pickup_datetime', 'lpep_dropoff_datetime']].isnull().any(axis=1).sum(),
    "Zero passengers": (df['passenger_count'] == 0).sum(),
    "Negative trip distance": (df['trip_distance'] < 0).sum(),
    "Negative fare amount": (df['fare_amount'] < 0).sum(),
    "Negative total amount": (df['total_amount'] < 0).sum(),
}
quality_df = pd.DataFrame(issue_counts.items(), columns=["issue_type", "record_count"])
quality_df.to_csv("python_output/data_quality_issues.csv", index=False)
log_time("Data Quality Check", start)

# --- 5. Daily Summary Table ---
start = time.time()
df['trip_date'] = df['lpep_pickup_datetime'].dt.date
summary_df = df.groupby('trip_date').agg(
    total_trips=('VendorID', 'count'),
    avg_distance=('trip_distance', 'mean'),
    max_distance=('trip_distance', 'max'),
    avg_duration_minutes=('trip_duration_minutes', 'mean'),
    avg_fare=('fare_amount', 'mean'),
    avg_tip=('tip_amount', 'mean'),
    total_revenue=('total_amount', 'sum'),
    active_vendors=('VendorID', 'nunique')
).reset_index()
summary_df.to_csv("python_output/trip_daily_summary.csv", index=False)
log_time("Trip Daily Summary", start)


Load CSV took 0.13 seconds | Memory: 360.23 MB | CPU: 18.80%

Data Cleaning and Transformation took 0.49 seconds | Memory: 353.03 MB | CPU: 23.20%

Save Cleaned CSV took 0.63 seconds | Memory: 323.73 MB | CPU: 11.80%

Data Quality Check took 0.01 seconds | Memory: 324.04 MB | CPU: 30.20%

Trip Daily Summary took 0.03 seconds | Memory: 325.89 MB | CPU: 28.40%



In [45]:
df.shape

(51100, 21)

In [49]:
print("\n--- Missing Values ---")
print(df.isnull().sum())

# Check for negative values in numeric columns
print("\n--- Negative Values ---")
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_cols:
    negative_count = (df[col] < 0).sum()
    if negative_count > 0:
        print(f"{col}: {negative_count} negative values")


--- Missing Values ---
VendorID                 0
lpep_pickup_datetime     0
lpep_dropoff_datetime    0
store_and_fwd_flag       0
RatecodeID               0
PULocationID             0
DOLocationID             0
passenger_count          0
trip_distance            0
fare_amount              0
extra                    0
mta_tax                  0
tip_amount               0
tolls_amount             0
improvement_surcharge    0
total_amount             0
payment_type             0
trip_type                0
congestion_surcharge     0
trip_duration_minutes    0
trip_date                0
dtype: int64

--- Negative Values ---


# Data Modeling

In [50]:
import pandas as pd
import numpy as np
import time
import psutil
import os

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

# --- Monitor ---
def log_stats(stage, start_time):
    elapsed = time.time() - start_time
    mem = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)
    print(f"{stage} took {elapsed:.2f}s | Memory: {mem:.2f} MB")

# --- 1. Load data ---
start = time.time()
df = pd.read_csv("python_output/taxi_trips_cleaned.csv")
log_stats("Load data", start)

# --- 2. Cast numeric columns to float ---
numeric_cols = [
    'passenger_count', 'trip_distance', 'extra', 'mta_tax',
    'tip_amount', 'tolls_amount', 'trip_duration_minutes',
    'improvement_surcharge', 'total_amount', 'congestion_surcharge'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# --- 3. Feature selection ---
start = time.time()
target = 'fare_amount'
drop_cols = ['vendor_id', 'pickup_time', 'dropoff_time', 'flag', target]
features = [col for col in df.columns if col not in drop_cols]

# Detect categorical columns
categorical_cols = df[features].select_dtypes(include='object').columns.tolist()
numerical_cols = [col for col in features if col not in categorical_cols]
log_stats("Feature selection", start)

# --- 5. Split data ---
start = time.time()
X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
log_stats("Train-test split", start)

# --- 6. Pipeline ---
start = time.time()

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# --- 7. Train ---
train_start = time.time()
pipeline.fit(X_train, y_train)
log_stats("Model training", train_start)

# --- 8. Predict ---
pred_start = time.time()
y_pred = pipeline.predict(X_test)
log_stats("Prediction", pred_start)

# --- 9. Evaluate ---
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f"\n📊 RMSE: {rmse:.2f}")
print(f"📈 R² score: {r2:.4f}")

# --- 10. Coefficients ---
lin_reg = pipeline.named_steps['regressor']
feature_names = (
    numerical_cols +
    list(pipeline.named_steps['preprocessor'].transformers_[1][1].get_feature_names_out(categorical_cols))
)
coefs = lin_reg.coef_

# Sort feature importances
importance = sorted(zip(feature_names, coefs), key=lambda x: abs(x[1]), reverse=True)
print("\nTop Feature Coefficients:")
for feat, coef in importance[:10]:
    print(f"{feat}: {coef:.4f}")

# --- 11. Memory Usage ---
mem = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)
print(f"\n🧠 Final memory usage: {mem:.2f} MB")


Load data took 0.12s | Memory: 357.31 MB
Feature selection took 0.01s | Memory: 359.52 MB
Train-test split took 0.02s | Memory: 354.60 MB
Model training took 0.44s | Memory: 350.22 MB
Prediction took 0.16s | Memory: 351.46 MB

📊 RMSE: 0.28
📈 R² score: 0.9996

Top Feature Coefficients:
lpep_pickup_datetime_2021-07-19 16:02:22: -3.0492
lpep_dropoff_datetime_2021-07-19 16:53:57: -3.0492
mta_tax: -1.7271
lpep_pickup_datetime_2021-07-28 16:16:46: -1.1901
lpep_dropoff_datetime_2021-07-28 17:26:53: -1.1901
lpep_pickup_datetime_2021-07-14 16:28:49: -1.1710
lpep_dropoff_datetime_2021-07-14 17:41:01: -1.1710
lpep_pickup_datetime_2021-07-02 16:25:06: -1.1397
lpep_dropoff_datetime_2021-07-02 17:39:53: -1.1397
lpep_pickup_datetime_2021-07-30 18:57:54: -1.1248

🧠 Final memory usage: 352.79 MB
